<div style="background:#1F3864;color:white;padding:14px 18px;border-radius:6px"><b>INTERNATIONAL UNIVERSITY – VNU-HCMC · SCHOOL OF COMPUTER SCIENCE AND ENGINEERING</b><br>IT171IU · Statistical Learning</div>

# Lab 2: Linear Regression

*Simple and multiple linear regression with statsmodels and ISLP: estimation, inference, prediction, diagnostics, interactions, non-linear terms and qualitative predictors*

**Chapter:** ISLP Chapter 3 · **Duration:** 150 minutes (3 periods) · **Runs on:** Google Colab (recommended), Jupyter Notebook / JupyterLab, VS Code, PyCharm

**How to use this notebook:** first *File → Save a copy in Drive*. Run the **Setup** cell, then read each text cell and run the code cells in order with **Shift + Enter**. Write your solutions in the cells marked ✍️ and answer the questions in the text cells.

> **Legend:** black = original ISLP lab content (re-formatted) · <span style="color:red">red = content added for IT171IU (theory, worked examples, exercises, setup & submission guide)</span>

In [ ]:
# Student information – fill in and run this cell first
STUDENT_NAME = "Your Full Name"
STUDENT_ID   = "ITDSIUxxxxx"
CLASS_GROUP  = "IT171IU – Lab Group xx"
print(f"Lab 2 | {STUDENT_NAME} | {STUDENT_ID} | {CLASS_GROUP}")

# <span style="color:red">1. Objectives</span>

<span style="color:red">After this lab you will be able to:</span>

- <span style="color:red">Fit simple and multiple linear regression models with `statsmodels` (`sm.OLS`) and build design matrices with `ISLP.models.ModelSpec`.</span>
- <span style="color:red">Interpret coefficients, standard errors, $t$-statistics, $p$-values, confidence intervals, $R^2$, RSE and the $F$-statistic.</span>
- <span style="color:red">Distinguish **confidence intervals** for the mean response from **prediction intervals** for a new observation.</span>
- <span style="color:red">Diagnose problems with residual plots, leverage statistics, outliers and variance inflation factors (VIF).</span>
- <span style="color:red">Add interaction terms, polynomial terms and transformations, and compare nested models with `anova_lm()`.</span>
- <span style="color:red">Encode qualitative predictors with dummy variables and interpret their coefficients.</span>

> 📌 <span style="color:red">**Note** — **Lab plan – 150 minutes (3 periods)**</span>
>
> | <span style="color:red">Time</span> | <span style="color:red">Activity</span> |
> |---|---|
> | <span style="color:red">0 – 20 min</span> | <span style="color:red">Theory review: least squares, standard errors, t- and F-tests, R², diagnostics (Section 2)</span> |
> | <span style="color:red">20 – 75 min</span> | <span style="color:red">Guided lab: simple and multiple regression on Boston, interactions, polynomials, qualitative predictors (Section 3)</span> |
> | <span style="color:red">75 – 90 min</span> | <span style="color:red">Extra worked examples: least squares "by hand", transformations and diagnostics</span> |
> | <span style="color:red">90 – 140 min</span> | <span style="color:red">Exercises 1 – 7 (Section 4); challenge exercises 8 – 9 if time allows</span> |
> | <span style="color:red">140 – 150 min</span> | <span style="color:red">Run all cells, save and submit through the Google Form (Section 5)</span> |

# <span style="color:red">2. Theory</span>

## <span style="color:red">2.1 Simple linear regression</span>

<span style="color:red">We assume an approximately linear relationship between a predictor $X$ and a response $Y$:</span>

$$\color{red}{Y = \beta_0 + \beta_1 X + \varepsilon .}$$

<span style="color:red">The **least-squares** estimates minimise the residual sum of squares $\mathrm{RSS} = \sum_{i=1}^n (y_i - \hat\beta_0 - \hat\beta_1 x_i)^2$:</span>

$$\color{red}{\hat\beta_1 = \frac{\sum_{i=1}^n (x_i-\bar x)(y_i-\bar y)}{\sum_{i=1}^n (x_i-\bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x .}$$

<span style="color:red">The **standard error** $\mathrm{SE}(\hat\beta_1)$ measures how much $\hat\beta_1$ would vary over repeated samples. An approximate 95% confidence interval is $\hat\beta_1 \pm 2\,\mathrm{SE}(\hat\beta_1)$. To test $H_0:\beta_1 = 0$ we use</span>

$$\color{red}{t = \frac{\hat\beta_1 - 0}{\mathrm{SE}(\hat\beta_1)},}$$

<span style="color:red">which has a $t$-distribution with $n-2$ degrees of freedom under $H_0$; a small $p$-value means we reject $H_0$.</span>

## <span style="color:red">2.2 Assessing the fit</span>

- <span style="color:red">**Residual standard error:** $\mathrm{RSE} = \sqrt{\mathrm{RSS}/(n-2)}$ – the typical size of a prediction error, in the units of $Y$.</span>
- <span style="color:red">**Coefficient of determination:** $R^2 = 1 - \mathrm{RSS}/\mathrm{TSS}$ with $\mathrm{TSS} = \sum_i (y_i - \bar y)^2$ – the proportion of variability in $Y$ explained by the model (between 0 and 1). In simple regression $R^2 = \mathrm{Cor}(X,Y)^2$.</span>

## <span style="color:red">2.3 Multiple linear regression</span>

<span style="color:red">With $p$ predictors, $Y = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p + \varepsilon$. The coefficient $\beta_j$ is the average change in $Y$ for a one-unit increase in $X_j$ **holding all other predictors fixed**. To test whether *any* predictor is useful ($H_0: \beta_1 = \dots = \beta_p = 0$) we use the **F-statistic**</span>

$$\color{red}{F = \frac{(\mathrm{TSS}-\mathrm{RSS})/p}{\mathrm{RSS}/(n-p-1)} .}$$

<span style="color:red">To compare a smaller model with $q$ coefficients set to zero against the full model, `anova_lm()` computes the partial F-statistic $\frac{(\mathrm{RSS}_0-\mathrm{RSS})/q}{\mathrm{RSS}/(n-p-1)}$.</span>

<span style="color:red">In matrix form the estimates are $\hat\beta = (X^\top X)^{-1} X^\top y$, where the first column of the design matrix $X$ is the intercept column of ones.</span>

## <span style="color:red">2.4 Extending the linear model</span>

| <span style="color:red">Extension</span> | <span style="color:red">Example term</span> | <span style="color:red">Remark</span> |
|---|---|---|
| <span style="color:red">Qualitative predictor</span> | <span style="color:red">`ShelveLoc` → dummies `ShelveLoc[Good]`, `ShelveLoc[Medium]`</span> | <span style="color:red">coefficients are differences from the *baseline* level</span> |
| <span style="color:red">Interaction</span> | <span style="color:red">`('lstat', 'age')` → $\beta_3\,\mathrm{lstat}\times\mathrm{age}$</span> | <span style="color:red">the effect of one predictor depends on the other; keep the main effects (*hierarchical principle*)</span> |
| <span style="color:red">Polynomial</span> | <span style="color:red">`poly('lstat', degree=2)`</span> | <span style="color:red">still a *linear* model in the coefficients</span> |
| <span style="color:red">Transformation</span> | <span style="color:red">$\log(X)$, $\sqrt{X}$, $\log(Y)$</span> | <span style="color:red">may fix non-linearity or non-constant variance</span> |

## <span style="color:red">2.5 Potential problems and diagnostics</span>

| <span style="color:red">Problem</span> | <span style="color:red">How to detect</span> | <span style="color:red">Typical remedy</span> |
|---|---|---|
| <span style="color:red">Non-linearity</span> | <span style="color:red">pattern in the plot of residuals vs fitted values</span> | <span style="color:red">add polynomial terms / transform $X$</span> |
| <span style="color:red">Correlated errors</span> | <span style="color:red">residuals tracking over time</span> | <span style="color:red">time-series models</span> |
| <span style="color:red">Non-constant variance</span> | <span style="color:red">funnel shape in the residual plot</span> | <span style="color:red">transform $Y$ ($\log Y$, $\sqrt{Y}$), weighted least squares</span> |
| <span style="color:red">Outliers</span> | <span style="color:red">studentized residuals with $</span> | <span style="color:red">r_i</span> | <span style="color:red">> 3$</span> | <span style="color:red">check the data; robust methods</span> |
| <span style="color:red">High leverage points</span> | <span style="color:red">leverage $h_i$ much larger than $(p+1)/n$</span> | <span style="color:red">check the data; refit without them</span> |
| <span style="color:red">Collinearity</span> | <span style="color:red">$\mathrm{VIF}(\hat\beta_j) = 1/(1-R^2_{X_j\mid X_{-j}}) > 5$–$10$</span> | <span style="color:red">drop or combine predictors, regularisation (Lab 5)</span> |

<span style="color:red">**Confidence vs prediction interval.** A confidence interval quantifies the uncertainty about the *average* response $f(x_0)$; a prediction interval for an individual $y_0 = f(x_0) + \varepsilon$ is always **wider** because it also includes the irreducible error $\varepsilon$.</span>

# 3. Instructions and Worked Examples

<span style="color:red">Work through this section **in order**: read each explanation, *predict* what a cell will print, then run it. The black text and code are the original ISLP Chapter 3 lab; red notes, checkpoints and extra worked examples were added for IT171IU.</span>

### <span style="color:red">Setup cell</span>

In [ ]:
# [ADDED]
# 0. Setup – run this cell first (installs missing packages and fetches data files).
# On Google Colab the first run takes about 1 minute. If Colab asks you to
# "Restart session", accept, then run this cell once more.
import sys, os, subprocess, importlib.util, urllib.request

def ensure(module, pip_name=None):
    """pip-install a package only if it cannot be imported yet."""
    if importlib.util.find_spec(module) is None:
        print(f"Installing {pip_name or module} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)

ensure("ISLP", "ISLP")

print("Setup complete – Python", sys.version.split()[0])

In [ ]:
# [ADDED] helper for the additional datasets
# Loader for the additional datasets of this course (folder "additional-data/").
# If the CSV file is not found next to the notebook, it is rebuilt from its public source,
# so the notebook also works on Colab without uploading anything.
import pandas as pd
def load_extra(name):
    for folder in ["additional-data", "../additional-data", "/content/additional-data",
                   "/content/drive/MyDrive/additional-data"]:
        path = os.path.join(folder, name + ".csv")
        if os.path.exists(path):
            return pd.read_csv(path)
    from sklearn import datasets as skd
    if name == "diabetes":
        d = skd.load_diabetes(as_frame=True, scaled=False); df = d.frame.rename(columns={"target": "progression"})
    elif name == "breast_cancer":
        d = skd.load_breast_cancer(as_frame=True); df = d.frame.copy()
        df["diagnosis"] = pd.Categorical.from_codes(d.target, ["malignant", "benign"]); df = df.drop(columns="target")
    elif name == "wine":
        d = skd.load_wine(as_frame=True); df = d.frame.rename(columns={"target": "cultivar"})
    elif name == "digits":
        d = skd.load_digits(as_frame=True); df = d.frame.rename(columns={"target": "digit"})
    elif name == "rossi":
        from lifelines.datasets import load_rossi; df = load_rossi()
    else:
        raise ValueError(f"unknown dataset {name!r}")
    return df

## 3.1 Importing packages
We import our standard libraries at this top
level.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

### New imports
Throughout this lab we will introduce new functions and libraries. However,
we will import them here to emphasize these are the new
code objects in this lab. Keeping imports near the top
of a notebook makes the code more readable, since scanning the first few
lines tells us what libraries are used.

In [ ]:
import statsmodels.api as sm

 We will provide relevant details about the
functions below as they are needed.

Besides importing whole modules, it is also possible
to import only a few items from a given module. This
will help keep the  *namespace* clean.
We will use a few specific objects from the `statsmodels` package
which we import here.

In [ ]:
from statsmodels.stats.outliers_influence \
     import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

As one of the import statements above is quite a long line, we inserted a line break `\` to
ease readability.

We will also use some functions written for the labs in this book in the `ISLP`
package.

In [ ]:
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)

> 📌 <span style="color:red">**Note** — **ISLP helpers used in this lab**</span>
>
> <span style="color:red">`load_data()` returns a data set as a pandas data frame; `ModelSpec` (imported as `MS`) builds design matrices (intercept, dummies, interactions, polynomials); `summarize()` prints a compact coefficient table; `poly()` creates orthogonal polynomial columns.</span>

### Inspecting Objects and Namespaces
The
function  `dir()`
provides a list of
objects in a namespace.

In [ ]:
dir()

 This shows you everything that `Python` can find at the top level.
There are certain objects like `__builtins__` that contain references to built-in
functions like `print()`.

Every python object has its own notion of
namespace, also accessible with `dir()`. This will include
both the attributes of the object
as well as any methods associated with it. For instance, we see `'sum'` in the listing for an
array.

In [ ]:
A = np.array([3,5,11])
dir(A)

 This indicates that the object `A.sum` exists. In this case it is a method
that can be used to compute the sum of the array `A` as can be seen by typing `A.sum?`.

In [ ]:
A.sum()

## 3.2 Simple Linear Regression
In this section we will  construct model
matrices (also called design matrices) using the `ModelSpec()`  transform from `ISLP.models`.

We  will use the `Boston` housing data set, which is contained in the `ISLP` package.  The `Boston` dataset records  `medv`  (median house value) for $506$ neighborhoods
around Boston.  We will build a regression model to predict  `medv`  using $13$
predictors such as  `rm`  (average number of rooms per house),
 `age`  (proportion of owner-occupied units built prior to 1940), and  `lstat`  (percent of
households with low socioeconomic status).  We will use `statsmodels` for this
task, a `Python` package that implements several commonly used
regression methods.

We have included a simple loading function `load_data()` in the
`ISLP` package:

In [ ]:
Boston = load_data("Boston")
Boston.columns

Type `Boston?` to find out more about these data.

We start by using the `sm.OLS()`  function to fit a
simple linear regression model.  Our response will be
 `medv`  and  `lstat`  will be the single predictor.
For this model, we can create the model matrix by hand.

In [ ]:
X = pd.DataFrame({'intercept': np.ones(Boston.shape[0]),
                  'lstat': Boston['lstat']})
X[:4]

We extract the response, and fit the model.

In [ ]:
y = Boston['medv']
model = sm.OLS(y, X)
results = model.fit()

Note that `sm.OLS()` does
not fit the model; it specifies the model, and then `model.fit()` does the actual fitting.

Our `ISLP` function `summarize()` produces a simple table of the parameter estimates,
their standard errors, t-statistics and p-values.
The function takes a single argument, such as the object `results`
returned here by the `fit`
method, and returns such a summary.

In [ ]:
summarize(results)

Before we describe other methods for working with fitted models, we outline a more useful and general framework for constructing a model matrix~`X`.
### Using Transformations: Fit and Transform
Our model above has a single predictor, and constructing `X` was straightforward.
In practice  we often fit models with more than one predictor, typically selected from an array or data frame.
We may wish to introduce transformations to the variables before fitting the model, specify interactions between variables, and expand some particular variables into sets of variables (e.g. polynomials).
The `sklearn`  package has a particular notion
for this type of task: a *transform*. A transform is an object
that is created with some parameters as arguments. The
object has two main methods: `fit()` and `transform()`.

We provide a general approach for specifying models and constructing
the model matrix through the transform `ModelSpec()` in the `ISLP` library.
`ModelSpec()`
(renamed `MS()` in the preamble) creates a
transform object, and then a pair of methods
`transform()` and `fit()` are used to construct a
corresponding model matrix.

We first describe this process for our simple regression model  using a single predictor `lstat` in
the `Boston` data frame, but will use it repeatedly in more
complex tasks in this and other labs in this book.
In our case the transform is created by the expression
`design = MS(['lstat'])`.

The `fit()`  method takes the original array and may do some
initial computations on it, as specified in the transform object.
For example, it may compute means and standard deviations for centering and scaling.
The `transform()`
method applies the fitted transformation to the array of data, and produces the model matrix.

In [ ]:
design = MS(['lstat'])
design = design.fit(Boston)
X = design.transform(Boston)
X[:4]

In this simple case, the `fit()`  method does very little; it simply checks that the variable `'lstat'` specified in `design` exists in `Boston`. Then `transform()` constructs the model matrix with two columns: an `intercept` and the variable `lstat`.

These two operations can be  combined with the
`fit_transform()`  method.

In [ ]:
design = MS(['lstat'])
X = design.fit_transform(Boston)
X[:4]

Note that, as in the previous code chunk when the two steps were done separately, the `design` object is changed as a result of the `fit()` operation. The power of this pipeline will become clearer when we fit more complex models that involve interactions and transformations.

Let's return to our fitted regression model.
The object
`results` has several methods that can be used for inference.
We already presented a function `summarize()` for showing the essentials of the fit.
For a full and somewhat exhaustive summary of the fit, we can use the `summary()`
method.

In [ ]:
results.summary()

The fitted coefficients can also be retrieved as the
`params` attribute of `results`.

In [ ]:
results.params

The `get_prediction()`  method can be used to obtain predictions, and produce confidence intervals and
prediction intervals for the prediction of  `medv`  for  given values of  `lstat`.

We first create a new data frame, in this case containing only the variable `lstat`, with the values for this variable at which we wish to make predictions.
We then use the `transform()` method of `design` to create the corresponding model matrix.

In [ ]:
new_df = pd.DataFrame({'lstat':[5, 10, 15]})
newX = design.transform(new_df)
newX

Next we compute the predictions at `newX`, and view them by extracting the `predicted_mean` attribute.

In [ ]:
new_predictions = results.get_prediction(newX);
new_predictions.predicted_mean

We can produce confidence intervals for the predicted values.

In [ ]:
new_predictions.conf_int(alpha=0.05)

Prediction intervals are computed by setting `obs=True`:

In [ ]:
new_predictions.conf_int(obs=True, alpha=0.05)

 For instance, the 95% confidence interval associated with an
 `lstat`  value of 10 is (24.47, 25.63), and the 95% prediction
interval is (12.82, 37.28).  As expected, the confidence and
prediction intervals are centered around the same point (a predicted
value of 25.05 for  `medv`  when  `lstat`  equals
10), but the latter are substantially wider.

Next we will plot  `medv`  and  `lstat`
using `DataFrame.plot.scatter()`, \definelongblankMR{plot.scatter()}{plot.slashslashscatter()}
and wish to
add the regression line to the resulting plot.

### Defining Functions
While there is a function
within the `ISLP` package that adds a line to an existing plot, we take this opportunity
to define our first function to do so.

In [ ]:
def abline(ax, b, m):
    "Add a line with slope m and intercept b to ax"
    xlim = ax.get_xlim()
    ylim = [m * xlim[0] + b, m * xlim[1] + b]
    ax.plot(xlim, ylim)

 A few things are illustrated above. First we see the syntax for defining a function:
`def funcname(...)`. The function has arguments `ax, b, m`
where `ax` is an axis object for an existing plot, `b` is the intercept and
`m` is the slope of the desired line. Other plotting  options can be passed on to
`ax.plot` by including additional optional arguments as follows:

In [ ]:
def abline(ax, b, m, *args, **kwargs):
    "Add a line with slope m and intercept b to ax"
    xlim = ax.get_xlim()
    ylim = [m * xlim[0] + b, m * xlim[1] + b]
    ax.plot(xlim, ylim, *args, **kwargs)

The addition of `*args` allows any number of
non-named arguments to `abline`, while `**kwargs` allows any
number of named arguments (such as `linewidth=3`) to `abline`.
In our function, we pass
these arguments verbatim to `ax.plot` above. Readers
interested in learning more about
functions are referred to the section on
defining functions in [docs.python.org/tutorial](https://docs.python.org/3/tutorial/controlflow.html#defining-functions).

Let’s use our new function to add this regression line to a plot of
`medv` vs. `lstat`.

In [ ]:
ax = Boston.plot.scatter('lstat', 'medv')
abline(ax,
       results.params['intercept'],
       results.params['lstat'],
       'r--',
       linewidth=3)

Thus, the final call to `ax.plot()` is `ax.plot(xlim, ylim, 'r--', linewidth=3)`.
We have used the argument `'r--'` to produce a red dashed line, and added
an argument to make it of width 3.
There is some evidence for non-linearity in the relationship between  `lstat`  and  `medv`. We will explore this issue later in this lab.

As mentioned above, there is an existing function to add a line to a plot --- `ax.axline()` --- but knowing how to write such functions empowers us to create more expressive displays.

Next we examine some diagnostic plots, several of which were discussed
in Section 3.3.3.
We can find the fitted values and residuals
of the fit as attributes of the `results` object.
Various influence measures describing the regression model
are computed with the `get_influence()` method.
As we will not use the `fig` component returned
as the first value from `subplots()`, we simply
capture the second returned value in `ax` below.

In [ ]:
ax = subplots(figsize=(8,8))[1]
ax.scatter(results.fittedvalues, results.resid)
ax.set_xlabel('Fitted value')
ax.set_ylabel('Residual')
ax.axhline(0, c='k', ls='--');

 We add a horizontal line at 0 for reference using the
 `ax.axhline()`   method, indicating
it should be black (`c='k'`) and have a dashed linestyle (`ls='--'`).

On the basis of the residual plot, there is some evidence of non-linearity.
Leverage statistics can be computed for any number of predictors using the
`hat_matrix_diag` attribute of the value returned by the
`get_influence()`  method.

In [ ]:
infl = results.get_influence()
ax = subplots(figsize=(8,8))[1]
ax.scatter(np.arange(X.shape[0]), infl.hat_matrix_diag)
ax.set_xlabel('Index')
ax.set_ylabel('Leverage')
np.argmax(infl.hat_matrix_diag)

 The `np.argmax()`  function identifies the index of the largest element of an array, optionally computed over an axis of the array.
In this case, we maximized over the entire array
to determine which observation has the largest leverage statistic.

> ✅ <span style="color:red">**Checkpoint** — **Simple linear regression**</span>
>
> 1. <span style="color:red">Write the fitted equation for `medv` on `lstat` and interpret the slope in words.</span>
> 2. <span style="color:red">What is the difference between the two intervals returned by `conf_int(alpha=0.05)` and `conf_int(obs=True, alpha=0.05)`?</span>
> 3. <span style="color:red">Which feature of the residual plot suggests that the relationship is not linear?</span>

## 3.3 Multiple Linear Regression
In order to fit a multiple linear regression model using least squares, we again use
the `ModelSpec()`  transform to construct the required
model matrix and response. The arguments
to `ModelSpec()` can be quite general, but in this case
a list of column names suffice. We consider a fit here with
the two variables `lstat` and `age`.

In [ ]:
X = MS(['lstat', 'age']).fit_transform(Boston)
model1 = sm.OLS(y, X)
results1 = model1.fit()
summarize(results1)

Notice how we have compacted the first line into a succinct expression describing the construction of `X`.

The  `Boston`   data set contains 12 variables, and so it would be cumbersome
to have to type all of these in order to perform a regression using all of the predictors.
Instead, we can use the following short-hand:\definelongblankMR{columns.drop()}{columns.slashslashdrop()}

In [ ]:
terms = Boston.columns.drop('medv')
terms

We can now fit the model with all the variables in `terms` using
the same model matrix builder.

In [ ]:
X = MS(terms).fit_transform(Boston)
model = sm.OLS(y, X)
results = model.fit()
summarize(results)

What if we would like to perform a regression using all of the variables but one?  For
example, in the above regression output,   `age`  has a high $p$-value.
So we may wish to run a regression excluding this predictor.
The following syntax results in a regression using all predictors except  `age`.

In [ ]:
minus_age = Boston.columns.drop(['medv', 'age']) 
Xma = MS(minus_age).fit_transform(Boston)
model1 = sm.OLS(y, Xma)
summarize(model1.fit())

## 3.4 Multivariate Goodness of Fit
We can access the individual components of `results` by name
(`dir(results)` shows us what is available). Hence
`results.rsquared` gives us the $R^2$,
and
`np.sqrt(results.scale)` gives us the RSE.

Variance inflation factors (section 3.3.3) are sometimes useful
to assess the effect of collinearity in the model matrix of a regression model.
We will compute the VIFs in our multiple regression fit, and use the opportunity to introduce the idea of *list comprehension*.

### List Comprehension
Often we encounter a sequence of objects which we would like to transform
for some other task. Below, we compute the VIF for each
feature in our `X` matrix and produce a data frame
whose index agrees with the columns of `X`.
The notion of list comprehension can often make such
a task easier.

List comprehensions are simple and powerful ways to form
lists of `Python` objects. The language also supports
dictionary and *generator* comprehension, though these are
beyond our scope here. Let's look at an example. We compute the VIF for each of the variables
in the model matrix `X`, using the function `variance_inflation_factor()`.

In [ ]:
vals = [VIF(X, i)
        for i in range(1, X.shape[1])]
vif = pd.DataFrame({'vif':vals},
                   index=X.columns[1:])
vif

The function `VIF()` takes two arguments: a dataframe or array,
and a variable column index. In the code above we call `VIF()` on the fly for all columns in `X`.
We have excluded column 0 above (the intercept), which is not of interest. In this case the VIFs are not that exciting.

The object `vals` above could have been constructed with the following for loop:

In [ ]:
vals = []
for i in range(1, X.values.shape[1]):
    vals.append(VIF(X.values, i))

List comprehension allows us to perform such repetitive operations in a more straightforward way.
## 3.5 Interaction Terms
It is easy to include interaction terms in a linear model using `ModelSpec()`.
Including a tuple `("lstat","age")` tells the model
matrix builder to include an interaction term between
 `lstat`  and  `age`.

In [ ]:
X = MS(['lstat',
        'age',
        ('lstat', 'age')]).fit_transform(Boston)
model2 = sm.OLS(y, X)
summarize(model2.fit())

## 3.6 Non-linear Transformations of the Predictors
The model matrix builder can include terms beyond
just column names and interactions. For instance,
the `poly()` function supplied in `ISLP` specifies that
columns representing polynomial functions
of its first argument are added to the model matrix.

In [ ]:
X = MS([poly('lstat', degree=2), 'age']).fit_transform(Boston)
model3 = sm.OLS(y, X)
results3 = model3.fit()
summarize(results3)

The effectively zero *p*-value associated with the quadratic term
(i.e. the third row above) suggests that it leads to an improved model.

By default, `poly()` creates a basis matrix for inclusion in the
model matrix whose
columns are *orthogonal polynomials*, which are designed for stable
least squares computations. {Actually, `poly()` is a  wrapper for the workhorse and standalone  function `Poly()` that does the  work in building the model matrix.}
Alternatively, had we included an argument
`raw=True` in the above call to `poly()`, the basis matrix would consist simply of
`lstat` and `lstat**2`. Since either of these bases
represent quadratic polynomials, the fitted values  would not
change in this case, just the polynomial coefficients.  Also by default, the columns
created by `poly()` do not include an intercept column as
that is automatically added by `MS()`.

We use the `anova_lm()` function to further quantify the extent to which the quadratic fit is
superior to the linear fit.

In [ ]:
anova_lm(results1, results3)

Here `results1` represents the linear submodel containing
predictors `lstat` and `age`,
while `results3` corresponds to the larger model above  with a quadratic
term in `lstat`.
The `anova_lm()` function performs a hypothesis test
comparing the two models. The null hypothesis is that the quadratic
term in the bigger model is not needed, and the alternative hypothesis is that the
bigger model is superior. Here the *F*-statistic is 177.28 and
the associated *p*-value is zero.
In this case the *F*-statistic is the square of the
*t*-statistic for the quadratic term in the linear model summary
for `results3` --- a consequence of the fact that these nested
models differ by one degree of freedom.
This provides very clear evidence that the quadratic polynomial in
`lstat` improves the linear model.
This is not surprising, since earlier we saw evidence for non-linearity in the relationship between `medv`
and  `lstat`.

The function `anova_lm()` can take more than two nested models
as input, in which case it compares every successive pair of models.
That also explains why there are `NaN`s in the first row above, since
there is no previous model with which to compare the first.

In [ ]:
ax = subplots(figsize=(8,8))[1]
ax.scatter(results3.fittedvalues, results3.resid)
ax.set_xlabel('Fitted value')
ax.set_ylabel('Residual')
ax.axhline(0, c='k', ls='--');

We see that when the quadratic term is included in the model,
there is little discernible pattern in the residuals.
In order to create a cubic or higher-degree polynomial fit, we can simply change the degree argument
to `poly()`.

> ✅ <span style="color:red">**Checkpoint** — **Multiple regression**</span>
>
> 1. <span style="color:red">Why is `age` significant in the simple model `medv ~ lstat + age` but not in the model with all predictors?</span>
> 2. <span style="color:red">Which predictors have a VIF above 5? What does this mean?</span>
> 3. <span style="color:red">What does the `anova_lm()` table tell you about the quadratic term in `lstat`?</span>

## 3.7 Qualitative Predictors
Here we use the  `Carseats`  data, which is included in the
`ISLP` package. We will  attempt to predict `Sales`
(child car seat sales) in 400 locations based on a number of
predictors.

In [ ]:
Carseats = load_data('Carseats')
Carseats.columns

The `Carseats`
 data includes qualitative predictors such as
 `ShelveLoc`, an indicator of the quality of the shelving
 location --- that is,
the  space within a store in which the car seat is displayed. The predictor
 `ShelveLoc`  takes on three possible values, `Bad`, `Medium`, and `Good`.
Given a qualitative variable such as  `ShelveLoc`, `ModelSpec()` generates dummy
variables automatically.
These variables are often referred to as a *one-hot encoding* of the categorical
feature. Their columns sum to one, so to avoid collinearity with an intercept, the first column is dropped. Below we see
the column `ShelveLoc[Bad]` has been dropped, since `Bad` is the first level of `ShelveLoc`.
Below we fit a multiple regression model that includes some interaction terms.

In [ ]:
allvars = list(Carseats.columns.drop('Sales'))
y = Carseats['Sales']
final = allvars + [('Income', 'Advertising'),
                   ('Price', 'Age')]
X = MS(final).fit_transform(Carseats)
model = sm.OLS(y, X)
summarize(model.fit())

In the first line above, we made `allvars` a list, so that we
could add the interaction terms two lines down.
Our model-matrix builder has created a `ShelveLoc[Good]`
dummy variable that takes on a value of 1 if the
shelving location is good, and 0 otherwise. It has also created a `ShelveLoc[Medium]`
dummy variable that equals 1 if the shelving location is medium, and 0 otherwise.
A bad shelving location corresponds to a zero for each of the two dummy variables.
The fact that the coefficient for `ShelveLoc[Good]` in the regression output is
positive indicates that a good shelving location is associated with high sales (relative to a bad location).
And `ShelveLoc[Medium]` has a smaller positive coefficient,
indicating that a medium shelving location leads to higher sales than a bad
shelving location, but lower sales than a good shelving location.

## <span style="color:red">3.8 Extra worked examples</span>

<span style="color:red">**Worked Example 1 – Least squares "by hand".** The coefficients reported by `statsmodels` are the solution of the *normal equations* $X^\top X\hat\beta = X^\top y$. We reproduce them with NumPy, together with the standard errors $\mathrm{SE}(\hat\beta_j) = \sqrt{\hat\sigma^2 [(X^\top X)^{-1}]_{jj}}$ and $R^2$.</span>

In [ ]:
# [ADDED] worked example – run this cell
X = MS(['lstat', 'age']).fit_transform(Boston)
y = Boston['medv']
Xn, yn = np.asarray(X), np.asarray(y)
n, p1 = Xn.shape                                   # p1 = p + 1 (intercept)
beta_hat = np.linalg.solve(Xn.T @ Xn, Xn.T @ yn)    # normal equations
resid = yn - Xn @ beta_hat
sigma2 = resid @ resid / (n - p1)                   # RSE^2
se = np.sqrt(sigma2 * np.diag(np.linalg.inv(Xn.T @ Xn)))
R2 = 1 - (resid @ resid) / np.sum((yn - yn.mean())**2)
by_hand = pd.DataFrame({'coef': beta_hat, 'std err': se, 't': beta_hat / se}, index=X.columns)
print(by_hand.round(4)); print('R^2 =', round(R2, 4))
print(summarize(sm.OLS(y, X).fit()))              # same numbers from statsmodels

<span style="color:red">**Worked Example 2 – A log transformation and diagnostic plots.** The residual plot of `medv ~ lstat` is curved. Replacing `lstat` by $\log(\mathrm{lstat})$ straightens the relationship; we compare $R^2$ and inspect studentized residuals and leverage.</span>

In [ ]:
# [ADDED] worked example – run this cell
Boston['log_lstat'] = np.log(Boston['lstat'])
fit_lin = sm.OLS(y, MS(['lstat']).fit_transform(Boston)).fit()
fit_log = sm.OLS(y, MS(['log_lstat']).fit_transform(Boston)).fit()
print(f"R^2 lstat: {fit_lin.rsquared:.3f}   R^2 log(lstat): {fit_log.rsquared:.3f}")

infl = fit_log.get_influence()
stud = infl.resid_studentized_external
lev = infl.hat_matrix_diag
fig, axes = subplots(1, 2, figsize=(11, 4))
axes[0].scatter(fit_log.fittedvalues, stud, s=10)
axes[0].axhline(3, c='r', ls='--'); axes[0].axhline(-3, c='r', ls='--')
axes[0].set_xlabel('fitted value'); axes[0].set_ylabel('studentized residual')
axes[1].scatter(lev, stud, s=10)
axes[1].axvline(3 * 2 / Boston.shape[0], c='r', ls='--')   # 3 x average leverage (p+1)/n
axes[1].set_xlabel('leverage'); axes[1].set_ylabel('studentized residual')
print('possible outliers (|r| > 3):', np.sum(np.abs(stud) > 3))
Boston = Boston.drop(columns='log_lstat')

<span style="color:red">*Interpretation:* the log model explains more variability ($R^2$ rises from about 0.54 to about 0.66). Points with $|r_i| > 3$ are candidate outliers; points far to the right in the second panel have high leverage and deserve a closer look before trusting the fit.</span>

# <span style="color:red">4. Exercises</span>

<span style="color:red">For every exercise write **code + output + a short interpretation**. Exercises 1–7 are compulsory; 8–9 are challenge (bonus) exercises. Use `load_data()` for the ISLP data sets and `load_extra()` (defined in the Setup section) for the additional data sets.</span>

## <span style="color:red">Exercise 1 – Simple regression on `Auto`</span>

- <span style="color:red">**(a)** Fit `mpg` on `horsepower` with `sm.OLS` and print the summary.</span>
- <span style="color:red">**(b)** Is there a relationship between the predictor and the response? How strong is it ($R^2$, RSE)? Is it positive or negative?</span>
- <span style="color:red">**(c)** What is the predicted `mpg` for `horsepower = 98`? Give the 95% **confidence** and **prediction** intervals.</span>
- <span style="color:red">**(d)** Plot `mpg` against `horsepower` with the least-squares line (use the `abline()` function of the lab) and draw the residual plot. Comment on any problems.</span>

In [ ]:
# ✍️ Exercise 1 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
Auto = load_data('Auto')
design_hp = MS(['horsepower'])
X_hp = design_hp.fit_transform(Auto)
fit_hp = None                    # TODO (a): sm.OLS(Auto['mpg'], X_hp).fit()
new = pd.DataFrame({'horsepower': [98]})
# TODO (c): pred = fit_hp.get_prediction(design_hp.transform(new)); pred.conf_int(alpha=0.05); pred.conf_int(obs=True, alpha=0.05)
# TODO (d): plots


## <span style="color:red">Exercise 2 – Multiple regression on `Auto`</span>

- <span style="color:red">**(a)** Produce the correlation matrix of the quantitative variables (exclude `name`).</span>
- <span style="color:red">**(b)** Fit `mpg` on **all** other variables except `name`. Which predictors are significant? What does the coefficient of `year` suggest?</span>
- <span style="color:red">**(c)** Draw the residual plot and the leverage plot. Are there unusually large outliers or high-leverage points?</span>
- <span style="color:red">**(d)** Compute the VIF of each predictor. Which ones indicate collinearity?</span>

In [ ]:
# ✍️ Exercise 2 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
Auto_num = Auto.drop(columns=['name']) if 'name' in Auto.columns else Auto.copy()
print(Auto_num.corr().round(2))
terms = Auto_num.columns.drop('mpg')
X_all = MS(terms).fit_transform(Auto_num)
fit_all = None                   # TODO (b)
# TODO (c), (d): use fit_all.get_influence() and VIF(X_all, i)


## <span style="color:red">Exercise 3 – Interactions and transformations</span>

<span style="color:red">Starting from Exercise 2, try **at least three** of the following and report $R^2$, the significance of the new terms and an `anova_lm()` comparison with the model without them:</span>

- <span style="color:red">an interaction such as `('horsepower', 'weight')` or `('displacement', 'year')`;</span>
- <span style="color:red">`poly('horsepower', degree=2)`;</span>
- <span style="color:red">$\log(\text{horsepower})$, $\sqrt{\text{horsepower}}$ or $\text{horsepower}^2$ (create new columns first).</span>

<span style="color:red">Which model would you choose and why?</span>

In [ ]:
# ✍️ Exercise 3 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
Auto_num['log_hp'] = np.log(Auto_num['horsepower'])
base = sm.OLS(Auto_num['mpg'], MS(['weight', 'year', 'horsepower']).fit_transform(Auto_num)).fit()
# TODO: inter = sm.OLS(Auto_num['mpg'], MS(['weight', 'year', 'horsepower', ('horsepower', 'weight')]).fit_transform(Auto_num)).fit()
# TODO: print(anova_lm(base, inter))


## <span style="color:red">Exercise 4 – Qualitative predictors with `Carseats`</span>

- <span style="color:red">**(a)** Fit `Sales` on `Price`, `Urban` and `US`.</span>
- <span style="color:red">**(b)** Interpret each coefficient – careful, `Urban` and `US` are qualitative.</span>
- <span style="color:red">**(c)** Write the model in equation form, with the dummy variables written out.</span>
- <span style="color:red">**(d)** For which predictors can you reject $H_0: \beta_j = 0$? Fit a smaller model with only those predictors and compare the two fits ($R^2$, RSE, `anova_lm`).</span>
- <span style="color:red">**(e)** Give 95% confidence intervals for the coefficients of the smaller model (`conf_int()`).</span>

In [ ]:
# ✍️ Exercise 4 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
Carseats = load_data('Carseats')
X_c = MS(['Price', 'Urban', 'US']).fit_transform(Carseats)
fit_c = None                     # TODO (a)
# TODO (d)–(e)


## <span style="color:red">Exercise 5 – Simulation: what does noise do to the fit?</span>

- <span style="color:red">**(a)** Using `rng = np.random.default_rng(1)`, create $x \sim N(0,1)$ with $n = 100$ and $\varepsilon \sim N(0, 0.25)$ (variance 0.25).</span>
- <span style="color:red">**(b)** Generate $y = -1 + 0.5x + \varepsilon$. What are $\beta_0$ and $\beta_1$ of the population model?</span>
- <span style="color:red">**(c)** Fit the least-squares line, compare $\hat\beta_0, \hat\beta_1$ with the truth and draw the data with both the fitted line and the population line (different colours, legend).</span>
- <span style="color:red">**(d)** Add $x^2$ to the model. Is there evidence that the quadratic term improves the fit?</span>
- <span style="color:red">**(e)** Repeat (a)–(c) with variance 0.05 and 1.0. How do the confidence intervals of $\beta_1$ change?</span>

In [ ]:
# ✍️ Exercise 5 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
rng = np.random.default_rng(1)
x = rng.normal(size=100)
eps = None                       # TODO (a): rng.normal(scale=np.sqrt(0.25), size=100)
y_sim = None                     # TODO (b)
# TODO (c)–(e)


## <span style="color:red">Exercise 6 – Collinearity</span>

- <span style="color:red">**(a)** Run the code in the starter cell: `y_col` depends on `x1` and `x2`, but `x2` is almost a copy of `x1`. Write the population model and its coefficients.</span>
- <span style="color:red">**(b)** What is the correlation between `x1` and `x2`? Draw a scatter plot.</span>
- <span style="color:red">**(c)** Fit `y_col` on `x1` and `x2`. Can you reject $H_0:\beta_1 = 0$? $H_0:\beta_2 = 0$? Compute the VIFs.</span>
- <span style="color:red">**(d)** Fit `y_col` on `x1` only and on `x2` only. Do the results contradict (c)? Explain.</span>

In [ ]:
# ✍️ Exercise 6 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
rng = np.random.default_rng(10)
x1 = rng.uniform(size=100)
x2 = 0.5 * x1 + rng.normal(size=100) / 10
y_col = 2 + 2 * x1 + 0.3 * x2 + rng.normal(size=100)
D = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y_col})
# TODO (b): np.corrcoef(x1, x2)
# TODO (c), (d): sm.OLS(D['y'], MS(['x1', 'x2']).fit_transform(D)).fit()


## <span style="color:red">Exercise 7 – Additional data set: diabetes progression</span>

<span style="color:red">The file `additional-data/diabetes.csv` contains 442 diabetes patients: age, sex, body-mass index (`bmi`), blood pressure (`bp`), six blood serum measurements `s1`–`s6` and the response `progression` (a quantitative measure of disease progression one year after baseline).</span>

- <span style="color:red">**(a)** Fit `progression` on `bmi` alone, then on `bmi`, `bp` and `s5`, then on all ten predictors. Report $R^2$ and RSE for the three models.</span>
- <span style="color:red">**(b)** Use `anova_lm()` to test whether the full model is significantly better than the three-predictor model.</span>
- <span style="color:red">**(c)** Compute the VIFs in the full model. Which serum measurements are strongly collinear?</span>
- <span style="color:red">**(d)** Which three predictors would you report to a doctor as most important? Justify with the output.</span>

In [ ]:
# ✍️ Exercise 7 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
diabetes = load_extra('diabetes')
print(diabetes.shape); print(diabetes.head())
y_d = diabetes['progression']
m1 = sm.OLS(y_d, MS(['bmi']).fit_transform(diabetes)).fit()
# TODO (a): m2 with ['bmi', 'bp', 's5'] and m3 with all predictors
# TODO (b): anova_lm(m2, m3)    (c): VIF


## <span style="color:red">Challenge exercises (optional, bonus)</span>

### <span style="color:red">Exercise 8 – Your own `ols()` function</span>

<span style="color:red">Write a function `ols(X, y)` that uses only NumPy to return a data frame with the coefficient, standard error, $t$-statistic and two-sided $p$-value of each column of `X` (use `scipy.stats.t.sf`), plus $R^2$ and the $F$-statistic. Check it against `sm.OLS` on the Boston model with all predictors.</span>

In [ ]:
# ✍️ Exercise 8 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
from scipy import stats
def ols(X, y):
    X, y = np.asarray(X, float), np.asarray(y, float)
    # TODO: beta, se, t, p-values, R2, F
    return None


### <span style="color:red">Exercise 9 – Regression through the origin</span>

<span style="color:red">Generate $x \sim N(0,1)$ and $y = 2x + \varepsilon$ ($n=100$). Fit $y$ on $x$ **without an intercept** (`MS(['x'], intercept=False)`) and $x$ on $y$ without an intercept. Show numerically that the two $t$-statistics are identical and explain why (textbook Exercise 3.11).</span>

In [ ]:
# ✍️ Exercise 9 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
rng = np.random.default_rng(1)
x9 = rng.normal(size=100)
y9 = 2 * x9 + rng.normal(size=100)
D9 = pd.DataFrame({'x': x9, 'y': y9})
# TODO: sm.OLS(D9['y'], MS(['x'], intercept=False).fit_transform(D9)).fit().tvalues


# <span style="color:red">5. Submission Instructions</span>

## <span style="color:red">5.1 What to submit</span>

<span style="color:red">Submit **one link** – to a **GitHub repository** (or sub-folder) **or** a **Google Drive folder** – that contains your lab files:</span>

1. <span style="color:red">**Notebook** `IT171IU_Lab02_FullName_StudentID.ipynb` – the completed notebook with **all cells executed** (*Runtime → Run all* before saving), all ✍️ cells filled in and the written answers typed in the text cells.</span>
2. <span style="color:red">**Report** `IT171IU_Lab02_Report.pdf` (or `.docx`) – this handout with the *Student Work Areas* filled in: short answers, key numbers and the most important plots/screenshots.</span>
3. <span style="color:red">**README.md** – your full name, student ID, class/group, how to run the notebook, and a note on any AI-assistant use (allowed only as stated in the course policy).</span>


<span style="color:red">*(added)*</span>
```text
IT171IU_Lab02_FullName_StudentID/
├── IT171IU_Lab02_FullName_StudentID.ipynb
├── IT171IU_Lab02_Report.pdf
├── figures/            (optional – exported plots)
└── README.md
```

## <span style="color:red">5.2 How to share the folder</span>

1. <span style="color:red">**GitHub (recommended):** create a repository such as `IT171IU-Statistical-Learning-Labs` with one sub-folder per lab, push your files and make the repository **Public** (or add the instructor as a collaborator).</span>
2. <span style="color:red">**Google Drive:** upload the folder → right-click → *Share* → *General access: Anyone with the link – Viewer* → *Copy link*.</span>
3. <span style="color:red">**Colab only (if everything is inside the notebook):** *Share* → *Anyone with the link – Viewer* → *Copy link*.</span>

<span style="color:red">*(added)*</span>
```bash
# first time (inside your labs folder)
git init
git add .
git commit -m "Lab 2 – FullName – StudentID"
git branch -M main
git remote add origin https://github.com/<your-username>/IT171IU-Statistical-Learning-Labs.git
git push -u origin main
# later labs:  git add . ;  git commit -m "Lab N" ;  git push
```

## <span style="color:red">5.3 Submit the link in the Google Form</span>

| <span style="color:red">Item</span> | <span style="color:red">Details</span> |
|---|---|
| <span style="color:red">Submission form for **Lab 2** (Lab Number pre-selected)</span> | <span style="color:red">[Open the Lab 2 submission form](https://docs.google.com/forms/d/e/1FAIpQLSeM7zOP9JarqtfBxE2RpljzppuZEfMPfygg7_k96zD5VlgZfQ/viewform?usp=pp_url&entry.1104082638=Lab+2:+Linear+Regression+(Ch.+3))</span> |
| <span style="color:red">General form link / short link</span> | <span style="color:red">[https://forms.gle/pUebamKV64bYB6KHA](https://forms.gle/pUebamKV64bYB6KHA) – choose *Lab 2: Linear Regression (Ch. 3)*</span> |
| <span style="color:red">Fields to fill in</span> | <span style="color:red">Full Name · Student ID · Class/Group · Lab Number · Submission Link/File</span> |
| <span style="color:red">Deadline</span> | <span style="color:red">End of the lab week – Sunday 23:59 (unless the instructor announces another date)</span> |

<span style="color:red">**Before you press Submit – checklist**</span>

- <span style="color:red">The link opens in a private/incognito browser window (sharing is set correctly).</span>
- <span style="color:red">The notebook runs from top to bottom without errors (*Runtime → Restart session and run all*).</span>
- <span style="color:red">Every exercise has code **and** a short written interpretation of the result.</span>
- <span style="color:red">File and folder names follow `IT171IU_Lab02_FullName_StudentID`.</span>